In [9]:
from astropy.table import Table
from astropy.constants import c

import numpy as np

from tqdm import tqdm

In [3]:
loa = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs_edited.fits')
loa[:5]

TARGETID,TARGET_RA,TARGET_DEC,Z,ZERR,SPECTYPE,DELTACHI2,ZWARN,PVTYPE,SGA_ID,PHOTSYS,DIST,DIST_R26,Selection,bad_fiber
int64,float64,float64,float64,float64,bytes6,float64,int64,bytes3,int64,bytes1,float64,float64,int64,int64
2350675715424256,239.8472,31.7866,0.1492904161251356,1.9422911736100356e-05,GALAXY,5071.658419655636,0,TFT,483369,S,0.00018163397694836428,0.04183588292464746,1,0
2389103635070977,61.51249486530825,-17.766542165450435,0.2898542763769689,1.3051647863939431e-05,GALAXY,585.0907674431801,0,TFT,526821,S,0.015034501051441454,0.40000000781105455,1,0
2389161461940225,53.110693190838745,-15.204162701018703,1.41029052508368,0.00015282405234457233,GALAXY,2.055874802172184,4,TFT,431568,S,0.020469853344286103,0.4000000051000554,1,0
2389161461940226,53.133949122267566,-15.238404516460593,1.490823662797069,8.773021438224206e-05,GALAXY,17.411476522684097,0,TFT,431568,S,0.020469853344283147,0.4000000050999976,1,0
2389190650101761,49.636675196299485,-14.026544506699297,0.01713651665060884,2.0408204503181013e-05,GALAXY,56.60990794678219,0,TFT,1074364,S,0.0026515899552215774,0.40000000679324754,1,0


In [7]:
c.to('km/s')

<Quantity 299792.458 km / s>

In [63]:
for i in tqdm(np.unique(loa['SGA_ID'])[:100]):

    # identify all fibers
    fiber = loa['SGA_ID'] == i
    
    # find all center fibers
    criteria = (loa['DIST_R26'][fiber] < .001) & (loa['ZWARN'][fiber] == 0) & (loa['DELTACHI2'][fiber] > 25)
    center = loa[fiber][criteria]

    if not np.any(center):
        continue

    # find z for all targets
    z_targ = loa['Z'][fiber]

    # find z for center targets
    z_c = center['Z']

    # calculate weight
    z_err = center['ZERR']
    weight = np.sum(1/ (z_err**2))

    # weighted z and error
    z_cen = np.nanmean((weight*z_c))/weight
    z_cen_err = np.sqrt(1/weight)

    # find the redshift of each fiber relative to the center
    z_rel = [(1 + z_targ)/(1 + z_cen) - 1 for j in z_targ]

    v = np.array(z_rel) * c.to('km/s')


  0%|          | 0/100 [00:00<?, ?it/s]


TypeError: Iterator operand 1 dtype could not be cast from dtype([('TARGETID', '<i8'), ('TARGET_RA', '<f8'), ('TARGET_DEC', '<f8'), ('Z', '<f8'), ('ZERR', '<f8'), ('SPECTYPE', 'S6'), ('DELTACHI2', '<f8'), ('ZWARN', '<i8'), ('PVTYPE', 'S3'), ('SGA_ID', '<i8'), ('PHOTSYS', 'S1'), ('DIST', '<f8'), ('DIST_R26', '<f8'), ('Selection', '<i8'), ('bad_fiber', '<i8')]) to dtype('bool') according to the rule 'unsafe'

In [66]:
#this works if there is a center point, but 
for i in np.unique(loa['SGA_ID'])[:100]:

    # identify all fibers
    fiber = loa['SGA_ID'] == i
    
    # find all center fibers
    criteria = (loa['DIST_R26'][fiber] < .001) & (loa['ZWARN'][fiber] == 0) & (loa['DELTACHI2'][fiber] > 25)
    center = loa[fiber][criteria]

    
    # find z for all targets
    z_targ = loa['Z'][fiber]

    # find z for center targets
    z_c = center['Z']

    # calculate weight
    z_err = center['ZERR']
    weight = np.sum(1/ (z_err**2))

    # weighted z and error
    z_cen = np.nanmean((weight*z_c))/weight
    z_cen_err = np.sqrt(1/weight)

    # print(z_cen)
    # print(z_cen_err)

    # find the redshift of each fiber relative to the center
    z_rel = (1 + z_targ)/(1 + z_cen) - 1

    # print(z_rel)

    v = np.array(z_rel) * c.to('km/s')
    # print(v)


/tmp/ipykernel_43364/1508554311.py:23: RuntimeWarning: Mean of empty slice
  z_cen = np.nanmean((weight*z_c))/weight
/tmp/ipykernel_43364/1508554311.py:24: RuntimeWarning: divide by zero encountered in scalar divide
  z_cen_err = np.sqrt(1/weight)
/tmp/ipykernel_43364/1508554311.py:23: RuntimeWarning: Mean of empty slice
  z_cen = np.nanmean((weight*z_c))/weight
/tmp/ipykernel_43364/1508554311.py:24: RuntimeWarning: divide by zero encountered in scalar divide
  z_cen_err = np.sqrt(1/weight)
/tmp/ipykernel_43364/1508554311.py:23: RuntimeWarning: Mean of empty slice
  z_cen = np.nanmean((weight*z_c))/weight
/tmp/ipykernel_43364/1508554311.py:24: RuntimeWarning: divide by zero encountered in scalar divide
  z_cen_err = np.sqrt(1/weight)
/tmp/ipykernel_43364/1508554311.py:23: RuntimeWarning: Mean of empty slice
  z_cen = np.nanmean((weight*z_c))/weight
/tmp/ipykernel_43364/1508554311.py:24: RuntimeWarning: divide by zero encountered in scalar divide
  z_cen_err = np.sqrt(1/weight)
/tmp/ipy